# The Transformer Mixed PPO Agent: A Complete Walkthrough

This notebook dissects every component of the **Transformer Mixed PPO** agent for
[Kaggle Orbit Wars](https://www.kaggle.com/competitions/orbit-wars). We use a
**real Kaggle environment** (not mock data) so you see exactly what the agent sees.

**What you'll learn:**
1. How raw Kaggle observations are parsed and encoded into tensors
2. The transformer policy architecture (~493K params)
3. Loss functions: BC, PPO, value, entropy
4. The DAgger pipeline (BC pretrain → PPO + imitation decay)
5. Dense relative reward and early production bonus
6. Mixed self-play and 4-player training curriculum

## Part 0: Setup

In [ ]:
#@title Install Dependencies & Clone Repo (Colab only)
#@markdown Run this cell if you're on Google Colab. Skip if running locally.

import subprocess, importlib, os
try:
    importlib.import_module("kaggle_environments")
    print("Dependencies already installed.")
except ImportError:
    subprocess.check_call([
        "pip", "install", "-q",
        "kaggle-environments>=1.28.0",
        "torch", "pyyaml", "numpy",
    ])
    if not os.path.exists("src"):
        subprocess.check_call(["git", "clone", "https://github.com/oscaraeschbacher/orbit_wars.git", "/tmp/ow"])
        os.chdir("/tmp/ow")
    print("Installed successfully.")

In [ ]:
import sys, os

# Ensure repo root is on path (handles both local and Colab)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import math
import numpy as np
import torch
import torch.nn.functional as F

from src.config import load_train_config
from src.game_types import parse_observation, PlanetState, FleetState, GameState
from src.features import (
    fleet_speed, passes_through_sun, compute_fleet_transit,
    encode_source_decision, GLOBAL_DIM, SOURCE_SCALAR_DIM,
    KNN_SCALAR_DIM, TARGET_SCALAR_DIM, FleetTransitState,
)
from src.policy import TransformerPolicy
from src.ppo import sample_actions, action_log_prob_and_entropy

# Load the mixed config
cfg = load_train_config('configs/transformer_mixed.yaml')
device = torch.device('cpu')

print(f"Config loaded: {cfg.run_name}")
print(f"  Reward mode: {cfg.reward.reward_mode}")
print(f"  4-player prob: {cfg.four_player_prob}")
print(f"  PPO updates: {cfg.ppo.total_updates}")
print(f"  BC expert: {cfg.imitation.bc_expert}")
print(f"  Rule-based decay: {cfg.rule_based_prob_start} -> {cfg.rule_based_prob_end} "
      f"over {cfg.rule_based_decay_updates} updates")

---

## Part 1: The Game State — What the Agent Sees

Before we touch any neural network, we need to understand **what the agent receives**.
Instead of mock data, let's create a **real Kaggle environment** and play a few turns.

In [ ]:
# Create a real Kaggle orbit_wars environment
from kaggle_environments import make

env = make("orbit_wars", debug=False)
env.reset(num_agents=2)

# Step once with empty moves to get initial observations
states = env.step([[], []])

# Import the apex agent so we can play a realistic mid-game
from agents.apex import agent as apex_agent

# Play ~60 turns of apex vs apex to get a mid-game state with fleets
for _ in range(60):
    p0_moves = apex_agent(states[0].observation)
    p1_moves = apex_agent(states[1].observation)
    states = env.step([p0_moves, p1_moves])

obs = states[0].observation
print(f"Step: {obs.step}")
print(f"Player: {obs.player}")
print(f"Planets: {len(obs.planets)}")
print(f"Fleets: {len(obs.fleets)}")
print(f"Angular velocity: {obs.angular_velocity}")
print(f"Remaining overage time: {obs.remainingOverageTime}s")

### 1.1 Raw Observation Fields

The Kaggle environment provides these fields. Let's inspect every one:

In [ ]:
# Show ALL raw observation fields
# Planets can be lists, dicts, or Struct objects depending on Kaggle version
# Helper to access fields regardless of format:
def _get(obj, key, idx):
    if hasattr(obj, key): return getattr(obj, key)
    if isinstance(obj, dict): return obj[key]
    return obj[idx]

print("=== Raw Observation Fields ===\n")

# Planets — format: [id, owner, x, y, radius, ships, production]
print(f"--- planets ({len(obs.planets)} entries) ---")
print(f"  Format: {type(obs.planets[0]).__name__}")
for p in obs.planets[:3]:
    pid = _get(p, 'id', 0)
    owner = _get(p, 'owner', 1)
    x, y = _get(p, 'x', 2), _get(p, 'y', 3)
    radius = _get(p, 'radius', 4)
    ships = _get(p, 'ships', 5)
    prod = _get(p, 'production', 6)
    print(f"  Planet {pid}: owner={owner}, pos=({x:.1f}, {y:.1f}), "
          f"radius={radius:.2f}, ships={ships}, prod={prod}")
print(f"  ... ({len(obs.planets)} total)\n")

# Fleets
n_fleets = len(obs.fleets) if obs.fleets else 0
print(f"--- fleets ({n_fleets} entries) ---")
if obs.fleets:
    print(f"  Format: {type(obs.fleets[0]).__name__}")
    for f in obs.fleets[:3]:
        fid = _get(f, 'id', 0)
        fowner = _get(f, 'owner', 1)
        fx, fy = _get(f, 'x', 2), _get(f, 'y', 3)
        fangle = _get(f, 'angle', 4)
        ffrom = _get(f, 'from_planet_id', 5)
        fships = _get(f, 'ships', 6)
        print(f"  Fleet {fid}: owner={fowner}, pos=({fx:.1f}, {fy:.1f}), "
              f"angle={fangle:.2f}rad, from_planet={ffrom}, ships={fships}")
    if n_fleets > 3:
        print(f"  ... ({n_fleets} total)")
else:
    print(f"  (no fleets yet)")
print()

# Comets
print(f"--- comets ---")
if obs.comets:
    for cg in obs.comets:
        n_c = len(cg.planets) if hasattr(cg, 'planets') else len(cg.get('planets', []))
        pi = cg.path_index if hasattr(cg, 'path_index') else cg.get('path_index', '?')
        print(f"  Group with {n_c} comets, path_index={pi}")
else:
    print(f"  None yet (spawn at steps 50, 150, 250, 350, 450)")
print(f"  comet_planet_ids: {obs.comet_planet_ids}\n")

# Other fields
print(f"--- other fields ---")
print(f"  angular_velocity: {obs.angular_velocity} rad/turn")
print(f"  step: {obs.step}")
print(f"  remainingOverageTime: {obs.remainingOverageTime}s")
print(f"  initial_planets: {len(obs.initial_planets)} entries (snapshot of starting positions)")

### 1.2 Observation Format Diversity

Kaggle observations can be **Struct objects** (attribute access like `p.owner`),
**dicts**, or **lists**. Our `parse_observation()` handles all three formats gracefully.

In [ ]:
# Show how parse_observation handles the raw Kaggle data
state = parse_observation(obs)

print("=== Parsed GameState ===")
print(f"Step: {state.step}/500  |  We are Player {state.player}")
print(f"Angular velocity: {state.angular_velocity:.4f} rad/turn")

print(f"\nPlanets ({len(state.planets)}):")
for p in state.planets:
    owner_str = {-1: 'NEUTRAL'}.get(p.owner, f'P{p.owner}')
    if p.owner == state.player:
        owner_str = 'OURS'
    elif p.owner >= 0:
        owner_str = 'ENEMY'
    orbit_str = ' [ORBITING]' if p.is_orbiting else ''
    print(f"  Planet {p.id:2d}: {owner_str:>7} at ({p.x:5.1f},{p.y:5.1f}) "
          f"ships={p.ships:3d} prod={p.production}{orbit_str}")

print(f"\nFleets ({len(state.fleets)}):")
for f in state.fleets[:8]:
    owner_str = 'OURS' if f.owner == state.player else f'P{f.owner}'
    print(f"  Fleet {f.id:2d}: {owner_str:>4} at ({f.x:5.1f},{f.y:5.1f}) "
          f"ships={f.ships:3d} angle={f.angle:.2f}rad")
if len(state.fleets) > 8:
    print(f"  ... ({len(state.fleets)} total)")

### 1.3 Orbiting Planet Detection

Inner planets orbit the sun. We detect this by comparing current positions against
`initial_planets` — if a planet's distance from center is less than 50 (board radius
minus planet radius), it's orbiting.

In [ ]:
# Show orbiting planet detection
orbiting = [p for p in state.planets if p.is_orbiting]
static = [p for p in state.planets if not p.is_orbiting]

print(f"=== Orbit Detection ===")
print(f"Orbiting planets: {len(orbiting)}")
for p in orbiting:
    dist_from_center = math.hypot(p.x - 50, p.y - 50)
    print(f"  Planet {p.id}: orbital_radius={p.orbital_radius:.1f}, "
          f"initial_angle={math.degrees(p.initial_angle):.1f}deg, "
          f"current_dist_from_center={dist_from_center:.1f}")
print(f"\nStatic planets: {len(static)}")
for p in static[:4]:
    dist_from_center = math.hypot(p.x - 50, p.y - 50)
    print(f"  Planet {p.id}: dist_from_center={dist_from_center:.1f}")
if len(static) > 4:
    print(f"  ... ({len(static)} total)")

### 1.4 Fleet Physics: Speed, Sun Avoidance, and Transit

Fleet speed is a nonlinear function of ship count. This is critical for strategy —
small scouting fleets are *slow* (1 unit/turn), while large fleets are fast (up to 6).

In [ ]:
# Fleet speed as a function of ship count
ship_counts = [1, 5, 10, 25, 50, 100, 200, 500, 1000]
print("Ships -> Speed (units/turn)")
print("-" * 35)
for s in ship_counts:
    spd = fleet_speed(s)
    bar = '#' * int(spd * 8)
    print(f"  {s:>5} ships -> {spd:.2f}  {bar}")

print(f"\nSun check: passes_through_sun(25, 25, 75, 75) = "
      f"{passes_through_sun(25, 25, 75, 75)}")
print(f"Sun check: passes_through_sun(25, 25, 75, 25) = "
      f"{passes_through_sun(25, 25, 75, 25)}")

### 1.5 Fleet Transit: Who's Going Where?

Before deciding where to send ships, the agent needs to know **what's already in
flight**. `compute_fleet_transit()` traces each fleet's trajectory to predict which
planet it will hit and when.

In [ ]:
transit = compute_fleet_transit(state)

print("=== Fleet Transit State ===")
print("(For each planet: incoming ships and estimated arrival time)\n")

entries = 0
for pid, info in sorted(transit.transit.items()):
    planet = state.planets_by_id[pid]
    owner_str = {-1: 'NEU'}.get(planet.owner, f'P{planet.owner}')
    if planet.owner == state.player:
        owner_str = 'OUR'
    parts = []
    if info.enemy_ships > 0:
        parts.append(f"enemy: {info.enemy_ships:.0f} ships in {info.enemy_eta:.1f}t")
    if info.friendly_ships > 0:
        parts.append(f"friendly: {info.friendly_ships:.0f} ships in {info.friendly_eta:.1f}t")
    if parts:
        print(f"  Planet {pid:2d} [{owner_str}]: {' | '.join(parts)}")
        entries += 1

if entries == 0:
    print("  (No fleets predicted to hit any planet)")

# Show how transit state gets updated sequentially
print("\n--- Sequential Transit Updates ---")
print("When planet A decides to send 20 ships to planet 6:")
transit.add_fleet(6, 20.0, 8.5, is_friendly=True)
info = transit.transit[6]
print(f"  Planet 6 transit: friendly={info.friendly_ships:.0f} ships, eta={info.friendly_eta:.1f}t")

print("When planet B ALSO sends 10 ships to planet 6:")
transit.add_fleet(6, 10.0, 5.0, is_friendly=True)
info = transit.transit[6]
print(f"  Planet 6 transit: friendly={info.friendly_ships:.0f} ships, eta={info.friendly_eta:.1f}t")
print(f"  (ETA is weighted average: (20*8.5 + 10*5.0) / 30 = {(20*8.5+10*5.0)/30:.1f}t)")

---

## Part 2: Feature Encoding — From Game State to Tensors

This is where domain knowledge meets neural networks. We convert the raw game state
into carefully engineered feature vectors.

**Decision order**: owned planets are processed in descending order of ship count.
The planet with the most ships decides first, then the next, etc. Each decision
updates the transit state so later planets see earlier launches.

In [ ]:
# Reset transit for clean demo
transit = compute_fleet_transit(state)

# Get our planets sorted by ship count (descending) — this is the decision order
my_planets = sorted(
    [p for p in state.planets if p.owner == state.player],
    key=lambda p: -p.ships
)
print("=== Decision Order (most ships first) ===")
for i, p in enumerate(my_planets):
    orbit_str = ' [ORBITING]' if p.is_orbiting else ''
    print(f"  Decision {i}: Planet {p.id} ({p.ships} ships, prod={p.production}){orbit_str}")

# Encode the FIRST decision (our strongest planet)
src = my_planets[0]
decision = encode_source_decision(src, state, transit, cfg.env)

print(f"\n=== SourceDecision for Planet {src.id} ===")
print(f"Source: Planet {decision.source_id} with {decision.remaining_ships} ships")
print(f"Number of valid targets: {sum(decision.target_mask) - 2} (+ CLS + NoOp)")
print(f"Target planet IDs: {decision.target_planet_ids}")

### 2.1 Global Features — The Big Picture in 9 Numbers

The global features capture **where we stand in the game**: game progress,
relative ship counts, and relative production across up to 3 enemy players.

In [ ]:
g = decision.global_features
labels = [
    'step / 500 (game progress)',
    'my_ships (normalized)',
    'enemy1_ships (normalized)',
    'enemy2_ships (normalized)',
    'enemy3_ships (normalized)',
    'my_production (normalized)',
    'enemy1_production (normalized)',
    'enemy2_production (normalized)',
    'enemy3_production (normalized)',
]

print(f"Global Features [{GLOBAL_DIM}]:")
print("-" * 55)
for i, (val, label) in enumerate(zip(g, labels)):
    bar = '#' * int(val * 30)
    print(f"  [{i}] {val:>6.3f} | {bar:<30} | {label}")

### 2.2 Source Planet Features — Where Are We Sending From?

For the planet making the decision, we encode its position (2D) and 7 scalar features:

In [ ]:
print("Source Position [2]:")
print(f"  x/100 = {decision.source_position[0]:.4f}  (raw: {decision.source_position[0]*100:.1f})")
print(f"  y/100 = {decision.source_position[1]:.4f}  (raw: {decision.source_position[1]*100:.1f})")

src_labels = [
    'radius / 5',
    'production / max_prod',
    'log1p(ships) / 10',
    'log1p(enemy_transit_ships) / 10',
    'enemy_transit_eta / 50',
    'log1p(friendly_transit_ships) / 10',
    'friendly_transit_eta / 50',
]

print(f"\nSource Scalars [{SOURCE_SCALAR_DIM}]:")
print("-" * 60)
for i, (val, label) in enumerate(zip(decision.source_scalars, src_labels)):
    bar = '#' * int(val * 30)
    print(f"  [{i}] {val:>6.3f} | {bar:<30} | {label}")

### 2.3 KNN Features — What's In the Neighborhood?

The K=3 nearest neighbors give the network **local context**: is our planet
surrounded by friends or enemies? Are nearby planets worth attacking?

In [ ]:
K = cfg.env.k_neighbors
print(f"KNN: {K} nearest neighbors to Planet {src.id}")
print("=" * 60)

knn_labels = ['radius/5', 'production/max', 'log1p(ships)/10', 'orbiting']
for i in range(K):
    print(f"\n  Neighbor {i}:")
    print(f"    Position: ({decision.knn_positions[i][0]*100:.1f}, "
          f"{decision.knn_positions[i][1]*100:.1f})")
    for j, (val, label) in enumerate(zip(decision.knn_scalars[i], knn_labels)):
        print(f"    [{j}] {val:.3f} ({label})")

print(f"\nThese are mean-pooled into a single [{cfg.model.embed_dim}]-dim vector")

### 2.4 Target Features — Every Possible Destination

The T=30 closest planets become candidate targets. Each target gets 11 scalar
features plus a 2D position, encoding ownership, distance, garrison, incoming
fleets, and orbital status.

In [ ]:
T = cfg.env.max_targets
n_valid = len(decision.target_planet_ids)
print(f"Targets: {n_valid} valid out of {T} slots")
print(f"Mask: {decision.target_mask.sum()} active (CLS + NoOp + {decision.target_mask.sum()-2} targets)\n")

tgt_labels = [
    'neutral', 'friendly', 'enemy', 'distance/100', 'log1p(ships)/10',
    'production/max', 'log1p(enemy_transit)/10', 'enemy_eta/50',
    'log1p(friendly_transit)/10', 'friendly_eta/50', 'orbiting',
]

for i in range(min(5, n_valid)):
    pid = decision.target_planet_ids[i]
    p = state.planets_by_id[pid]
    owner_str = {-1: 'NEUTRAL'}.get(p.owner, 'OURS' if p.owner == state.player else 'ENEMY')
    orbit_str = ' [ORB]' if p.is_orbiting else ''
    print(f"  Target {i} -> Planet {pid} ({owner_str}{orbit_str})")
    for j, (val, label) in enumerate(zip(decision.target_scalars[i], tgt_labels)):
        if val != 0:
            print(f"    [{j:2d}] {val:.3f} ({label})")
    print()

if n_valid > 5:
    print(f"  ... ({n_valid} total targets)")

### 2.5 Complete Feature Summary

In [ ]:
print("=== Complete Feature Shapes (one SourceDecision) ===")
print(f"  global_features:    {decision.global_features.shape}   = {GLOBAL_DIM} scalars")
print(f"  source_position:    {decision.source_position.shape}   = (x, y)")
print(f"  source_scalars:     {decision.source_scalars.shape}   = {SOURCE_SCALAR_DIM} scalars")
print(f"  knn_positions:      {decision.knn_positions.shape}  = {K} neighbors x 2")
print(f"  knn_scalars:        {decision.knn_scalars.shape}  = {K} neighbors x {KNN_SCALAR_DIM}")
print(f"  target_positions:   {decision.target_positions.shape} = {T} targets x 2")
print(f"  target_scalars:     {decision.target_scalars.shape} = {T} targets x {TARGET_SCALAR_DIM}")
print(f"  target_mask:        {decision.target_mask.shape}  = CLS + NoOp + {T} targets")
print(f"\n  Total feature dimensions per decision: "
      f"{GLOBAL_DIM + 2 + SOURCE_SCALAR_DIM + K*(2+KNN_SCALAR_DIM) + T*(2+TARGET_SCALAR_DIM) + T+2}")

---

## Part 3: The Transformer Policy — From Features to Actions

Now for the main event. Let's instantiate the policy network and watch data flow
through every layer.

The architecture has 5 stages:

```
Raw Features
    -> Encoders (5 MLPs, one shared position encoder)
        -> Token Construction (CLS + NoOp + Targets)
            -> Transformer (2 layers, 4 heads)
                -> Output Heads (target logits, fraction logits, value)
```

> **Architecture philosophy**: This is NOT a standard sequence model. The transformer
> here acts as a **comparison engine** — targets attend to each other, learning to
> suppress inferior options and amplify the best one.

In [ ]:
# Create the policy
policy = TransformerPolicy(cfg.model, cfg.env).to(device)
policy.eval()  # no dropout

# Count parameters by component
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("=== TransformerPolicy Architecture ===")
print(f"Total parameters: {count_params(policy):,}\n")

components = [
    ('pos_encoder (shared)', policy.pos_encoder),
    ('global_encoder', policy.global_encoder),
    ('source_encoder', policy.source_encoder),
    ('knn_encoder', policy.knn_encoder),
    ('target_encoder', policy.target_encoder),
    ('token_projection', policy.token_projection),
    ('transformer_blocks', policy.transformer_blocks),
    ('target_head', policy.target_head),
    ('fraction_head', policy.fraction_head),
    ('value_head', policy.value_head),
]

for name, mod in components:
    n = count_params(mod)
    bar = '#' * max(1, n // 5000)
    print(f"  {name:<25} {n:>8,} params  {bar}")

### 3.1 Stage 1: The Encoders — Projecting to Embedding Space

Each input type gets its own MLP encoder that projects it into the shared
`embed_dim=128` space. Let's trace through each one.

**The Position Encoder** is particularly elegant: it's a shared 2-layer MLP
`(2 -> 64 -> 128)` that converts (x, y) coordinates into a learned spatial
embedding. The same encoder is used for source, KNN, and target positions.

> **Why share the position encoder?** Transfer learning within the model.
> A planet at (25, 75) should have the same spatial representation whether it's
> the source, a neighbor, or a target. This reduces parameters and helps generalization.

In [ ]:
# Prepare tensors (batch size 1)
B = 1
global_t = torch.tensor(decision.global_features, dtype=torch.float32).unsqueeze(0)
src_scalars_t = torch.tensor(decision.source_scalars, dtype=torch.float32).unsqueeze(0)
src_pos_t = torch.tensor(decision.source_position, dtype=torch.float32).unsqueeze(0)
knn_scalars_t = torch.tensor(decision.knn_scalars, dtype=torch.float32).unsqueeze(0)
knn_pos_t = torch.tensor(decision.knn_positions, dtype=torch.float32).unsqueeze(0)
tgt_scalars_t = torch.tensor(decision.target_scalars, dtype=torch.float32).unsqueeze(0)
tgt_pos_t = torch.tensor(decision.target_positions, dtype=torch.float32).unsqueeze(0)
mask_t = torch.tensor(decision.target_mask, dtype=torch.bool).unsqueeze(0)

T = cfg.env.max_targets
K = cfg.env.k_neighbors
d = cfg.model.embed_dim

with torch.no_grad():
    # 1a. Position encoding
    src_pos_emb = policy.pos_encoder(src_pos_t)  # [1, 128]
    print(f"1a. Position Encoder:")
    print(f"    Input: source position {list(src_pos_t.shape)}")
    print(f"    Output: {list(src_pos_emb.shape)}")
    print(f"    Values: [{src_pos_emb[0,:4].tolist()}...]\n")

    # 1b. Global encoding
    global_emb = policy.global_encoder(global_t)  # [1, 128]
    print(f"1b. Global Encoder:")
    print(f"    Input: global features {list(global_t.shape)}")
    print(f"    Output: {list(global_emb.shape)}\n")

    # 1c. Source encoding (concat pos_emb + scalars -> MLP)
    src_input = torch.cat([src_pos_emb, src_scalars_t], dim=-1)  # [1, 135]
    src_emb = policy.source_encoder(src_input)  # [1, 128]
    print(f"1c. Source Encoder:")
    print(f"    Input: concat(pos_emb[128], scalars[{SOURCE_SCALAR_DIM}]) = {list(src_input.shape)}")
    print(f"    Output: {list(src_emb.shape)}\n")

    # 1d. Source + global -> src_knn (combined context)
    src_knn = src_emb + global_emb  # element-wise sum
    print(f"    source_knn = source_emb + global_emb: {list(src_knn.shape)}")

In [ ]:
with torch.no_grad():
    # KNN encoding + mean pooling
    knn_pos_flat = knn_pos_t.reshape(B * K, 2)
    knn_pos_emb = policy.pos_encoder(knn_pos_flat).reshape(B, K, -1)  # [1, 3, 128]
    knn_input = torch.cat([knn_pos_emb, knn_scalars_t], dim=-1)  # [1, 3, 132]
    knn_emb = policy.knn_encoder(knn_input)  # [1, 3, 128]
    knn_pool = knn_emb.mean(dim=1)  # [1, 128] <-- mean over 3 neighbors

    print(f"1d. KNN Encoder:")
    print(f"    Per-neighbor: concat(pos_emb[128], scalars[{KNN_SCALAR_DIM}]) = [3, 132]")
    print(f"    After MLP: {list(knn_emb.shape)}")
    print(f"    After mean-pool: {list(knn_pool.shape)}")
    print(f"    -> Added to src_knn for combined context\n")

    src_knn = src_knn + knn_pool  # now [1, 128]

    # 1e. Target encoding
    tgt_pos_flat = tgt_pos_t.reshape(B * T, 2)
    tgt_pos_emb = policy.pos_encoder(tgt_pos_flat).reshape(B, T, -1)  # [1, 30, 128]
    tgt_input = torch.cat([tgt_pos_emb, tgt_scalars_t], dim=-1)  # [1, 30, 139]
    tgt_emb = policy.target_encoder(tgt_input)  # [1, 30, 128]

    print(f"1e. Target Encoder:")
    print(f"    Per-target: concat(pos_emb[128], scalars[{TARGET_SCALAR_DIM}]) = [30, 139]")
    print(f"    After MLP: {list(tgt_emb.shape)}")

### 3.2 Stage 2: Token Construction — Building the Sequence

Now comes the key architectural idea. We construct a **sequence of tokens** for
the transformer:

```
[CLS] [NoOp] [Target_0] [Target_1] ... [Target_29] [PAD...]
```

- **CLS**: A learnable parameter (like BERT's [CLS]). Used only for the value head.
- **NoOp**: A learnable parameter representing "send nothing." Competes with targets.
- **Targets**: Each target token is a projection of `concat(global, source_knn, target)` —
  this means every target sees the global state and the source planet's context.
- **PAD**: Zero-filled positions beyond valid targets. Masked out in attention.

In [ ]:
with torch.no_grad():
    # Broadcast global + source_knn into each target position
    global_exp = global_emb.unsqueeze(1).expand(B, T, d)     # [1, 30, 128]
    source_exp = src_knn.unsqueeze(1).expand(B, T, d)        # [1, 30, 128]

    # Concatenate: each target sees global + source_knn + its own features
    token_input = torch.cat([global_exp, source_exp, tgt_emb], dim=-1)  # [1, 30, 384]

    # Project down to embed_dim
    target_tokens = policy.token_projection(token_input)  # [1, 30, 128]

    # Build full sequence: [CLS, NoOp, Target_0, ..., Target_{T-1}]
    cls_token = policy.cls_token.unsqueeze(0).expand(B, 1, d)
    noop_token = policy.noop_token.unsqueeze(0).expand(B, 1, d)
    tokens = torch.cat([cls_token, noop_token, target_tokens], dim=1)  # [1, 32, 128]

    print(f"=== Stage 2: Token Construction ===")
    print(f"  target_tokens = projection(concat(global, src_knn, target_emb))")
    print(f"  Input to projection: {list(token_input.shape)}")
    print(f"  Target tokens: {list(target_tokens.shape)}")
    print(f"  Full sequence: {list(tokens.shape)}")
    print(f"    [0] = CLS (learnable)")
    print(f"    [1] = NoOp (learnable)")
    print(f"    [2:{2+len(decision.target_planet_ids)}] = valid targets")
    print(f"    [{2+len(decision.target_planet_ids)}:{T+2}] = padding (masked out)")
    print(f"  Mask shape: {list(mask_t.shape)} ({mask_t.sum().item()} active)")

### 3.3 Stage 3: The Transformer — Attention Is All You Need (for targeting)

The 2-layer transformer with 4 attention heads processes the token sequence. This
is where the magic happens: targets can attend to each other, learning patterns like:

- "If the best target is a high-production enemy planet, suppress interest in low-production neutrals"
- "If multiple targets are along the same heading, maybe only send to the nearest one"
- "If the NoOp token attends strongly to the source, keep ships for defense"

In [ ]:
with torch.no_grad():
    # Prepare attention mask
    key_padding_mask = ~mask_t  # True = masked out (MHA convention)

    # TransformerBlock uses batch_first=True
    x = tokens  # [1, 32, 128]

    print("=== Stage 3: Transformer Processing ===")
    print(f"Input shape: {list(x.shape)} (batch={x.shape[0]}, seq_len={x.shape[1]}, dim={x.shape[2]})")
    print(f"Attention mask: {key_padding_mask.shape} ({key_padding_mask.sum().item()} positions masked)\n")

    for i, block in enumerate(policy.transformer_blocks):
        x_before = x.clone()
        x = block(x, key_padding_mask=key_padding_mask)
        delta = (x - x_before).abs().mean().item()
        print(f"  Layer {i}: shape={list(x.shape)}, avg delta={delta:.6f}")

    transformer_out = x
    print(f"\nOutput: {list(transformer_out.shape)}")

### 3.4 Visualizing Attention Patterns

Let's peek inside the transformer's attention heads to see what relationships it learns.
Even with random weights, the mask structure is informative.

In [ ]:
# Extract attention weights from the first layer
with torch.no_grad():
    first_block = policy.transformer_blocks[0]
    x_in = tokens
    x_normed = first_block.ln1(x_in)
    attn_out, attn_weights = first_block.attn(
        x_normed, x_normed, x_normed,
        key_padding_mask=~mask_t,
        need_weights=True,
        average_attn_weights=False,
    )

# attn_weights: [B, n_heads, seq_len, seq_len]
n_heads = cfg.model.n_heads
n_valid_tokens = int(mask_t.sum().item())

print(f"Attention weights shape: {list(attn_weights.shape)}")
print(f"  {n_heads} heads, {n_valid_tokens} valid tokens\n")

# Show attention from CLS and NoOp to targets
labels = ['CLS', 'NoOp'] + [f'T{i}' for i in range(n_valid_tokens - 2)]
for head in range(n_heads):
    print(f"--- Head {head} ---")
    for src_idx, src_name in enumerate(['CLS', 'NoOp']):
        weights = attn_weights[0, head, src_idx, :n_valid_tokens].numpy()
        top_k = min(5, len(weights))
        top_indices = np.argsort(weights)[-top_k:][::-1]
        top_str = ", ".join(f"{labels[j]}={weights[j]:.2f}" for j in top_indices if j < len(labels))
        print(f"  {src_name} attends to: {top_str}")
    print()

### 3.5 Stage 4: Output Heads — From Representations to Decisions

Three output heads decode the transformer's representations:

1. **Target Head**: Linear projection `[128] -> [1]` on tokens 1..T+1, producing
   logits for NoOp + each target. Invalid targets get `-inf`.
2. **Fraction Head**: Linear projection `[128] -> [5]` on target tokens only,
   producing logits over 5 fraction bins `[0.2, 0.4, 0.6, 0.8, 1.0]`.
3. **Value Head**: MLP `[128] -> [128] -> [1]` on the CLS token, estimating
   expected return.

In [ ]:
with torch.no_grad():
    # Run the full forward pass through the policy
    outputs = policy(
        global_t, src_scalars_t, src_pos_t,
        knn_scalars_t, knn_pos_t,
        tgt_scalars_t, tgt_pos_t,
        mask_t
    )

print("=== Stage 4: Output Heads ===")
print(f"\nTarget logits shape: {list(outputs.target_logits.shape)} (NoOp + {T} targets)")
print(f"Fraction logits shape: {list(outputs.fraction_logits.shape)} ({T} targets x 5 fractions)")
print(f"Value estimate: {outputs.value.item():.4f}")

# Show target logits (only valid ones)
valid_logits = outputs.target_logits[0].numpy()
n_valid_targets = sum(decision.target_mask) - 2  # exclude CLS, NoOp
print(f"\n--- Target Logits (valid only) ---")
print(f"  NoOp: {valid_logits[0]:.4f}")
for i in range(min(8, n_valid_targets)):
    pid = decision.target_planet_ids[i]
    p = state.planets_by_id[pid]
    owner = {-1: 'N'}.get(p.owner, 'O' if p.owner == state.player else 'E')
    print(f"  Target {i} (P{pid} [{owner}]): {valid_logits[i+1]:.4f}")

In [ ]:
# Show fraction logits for the top target
top_target = int(np.argmax(valid_logits))

print(f"--- Fraction Logits (for top target: index {top_target}) ---")
if top_target == 0:
    print("Top action is NoOp -- fraction is irrelevant (zeroed out)")
else:
    frac_logits = outputs.fraction_logits[0, top_target - 1].numpy()  # [5]
    frac_probs = np.exp(frac_logits - frac_logits.max())
    frac_probs = frac_probs / frac_probs.sum()

    fractions = cfg.env.ship_fractions
    src_ships = src.ships
    for i, (frac, prob, logit) in enumerate(zip(fractions, frac_probs, frac_logits)):
        ships = int(src_ships * frac)
        bar = '#' * int(prob * 40)
        print(f"  Bin {i} ({frac:>4.0%} = {ships:>3} ships): logit={logit:+.3f}  prob={prob:.3f} {bar}")

### 3.6 Sampling Actions — From Distributions to Moves

Now let's see how we go from logits to actual game moves. The `sample_actions`
function creates two Categorical distributions and samples from them.

**The factored action decomposition**:
```
log_prob(action) = log_prob(target) + log_prob(fraction | target)
entropy(action) = entropy(target) + entropy(fraction)
```

This is valid because we treat target and fraction as independent (conditional
independence given the transformer output).

In [ ]:
with torch.no_grad():
    # Stochastic sampling (for training)
    sampled = sample_actions(outputs, deterministic=False)

    # Deterministic (argmax) for evaluation
    sampled_det = sample_actions(outputs, deterministic=True)

print("=== Action Sampling ===")
print(f"\nStochastic sample (for training):")
print(f"  target_index: {sampled.target_index.item()} "
      f"({'NoOp' if sampled.target_index.item() == 0 else f'Planet {decision.target_planet_ids[sampled.target_index.item()-1]}'})")
print(f"  fraction_bin: {sampled.fraction_bin.item()} "
      f"({cfg.env.ship_fractions[sampled.fraction_bin.item()]:.0%})")
print(f"  log_prob: {sampled.log_prob.item():.4f}")
print(f"  entropy: {sampled.entropy.item():.4f}")

print(f"\nDeterministic (for evaluation):")
print(f"  target_index: {sampled_det.target_index.item()} "
      f"({'NoOp' if sampled_det.target_index.item() == 0 else f'Planet {decision.target_planet_ids[sampled_det.target_index.item()-1]}'})")
print(f"  fraction_bin: {sampled_det.fraction_bin.item()} "
      f"({cfg.env.ship_fractions[sampled_det.fraction_bin.item()]:.0%})")

### 3.7 Full Forward Pass — End-to-End Summary

Let's trace the complete data flow one more time, now showing tensor shapes at
every step:

In [ ]:
print("=" * 70)
print("COMPLETE FORWARD PASS: Observation -> Action")
print("=" * 70)

n_valid_targets = sum(decision.target_mask) - 2
steps = [
    ("Raw observation", "Kaggle env", "planets, fleets, step, ..."),
    ("parse_observation()", "-> GameState", f"{len(state.planets)} planets, {len(state.fleets)} fleets"),
    ("compute_fleet_transit()", "-> FleetTransitState", f"{len(transit.transit)} planet transit entries"),
    ("encode_source_decision()", "-> SourceDecision",
     f"global[{GLOBAL_DIM}] + src[2+{SOURCE_SCALAR_DIM}] + knn[{K}x{2+KNN_SCALAR_DIM}] + tgt[{T}x{2+TARGET_SCALAR_DIM}]"),
    ("Encoders (5 MLPs)", "-> embeddings", f"global[{d}] + src_knn[{d}] + targets[{T},{d}]"),
    ("Token construction", f"-> [{T+2},{d}]", f"CLS + NoOp + {n_valid_targets} targets + {T-n_valid_targets} pad"),
    ("Transformer (2 layers)", f"-> [{T+2},{d}]", f"self-attention over {n_valid_targets+2} valid tokens"),
    ("Target head", f"-> [{T+1}]", f"logits for NoOp + {T} targets ({n_valid_targets} valid)"),
    ("Fraction head", f"-> [{T},5]", f"5-bin fraction per target"),
    ("Value head", "-> [1]", f"expected return: {outputs.value.item():.4f}"),
    ("sample_actions()", "-> SampledAction", f"target={sampled_det.target_index.item()}, frac={sampled_det.fraction_bin.item()}"),
]

for i, (name, shape, detail) in enumerate(steps):
    arrow = "  ->" if i > 0 else "   "
    print(f"{arrow} {name:<30} {shape:<20} {detail}")

---

## Part 4: The Loss Functions — How the Agent Learns

We have three learning signals, used at different phases:

1. **Behavioral Cloning (BC) loss** — Cross-entropy against expert demonstrations
2. **PPO policy loss** — Clipped surrogate objective from RL
3. **Value loss** — MSE between predicted and actual returns

During DAgger training, the total loss is:
```
loss = policy_loss + 0.5 * value_loss - 0.01 * entropy + beta * bc_loss
```
where `beta` decays linearly from 0.5 to 0 over the first 1000 updates.

> **Why three losses?** Each serves a different purpose:
> - BC loss says "play like the expert" (fast but limited by expert quality)
> - PPO loss says "improve based on trial-and-error" (slow but unbounded)
> - Value loss trains the critic, which PPO needs for advantage estimation
> - Entropy bonus prevents premature convergence to a single strategy

In [ ]:
print("=== Loss Function Anatomy ===")
print()

# --- 1. BC Loss (Behavioral Cloning) ---
# Suppose the expert chose target_index=2 (first real target) and fraction_bin=3 (80%)
expert_target = torch.tensor([2], dtype=torch.long)
expert_frac = torch.tensor([3], dtype=torch.long)

# Target cross-entropy
safe_logits = outputs.target_logits.clamp(min=-1e4)  # avoid -inf issues
target_ce = F.cross_entropy(safe_logits, expert_target)

# Fraction cross-entropy (only for non-NoOp)
frac_idx = (expert_target - 1).clamp(min=0)  # offset for fraction indexing
frac_logits_sel = outputs.fraction_logits[0, frac_idx[0]].unsqueeze(0)  # [1, 5]
frac_ce = F.cross_entropy(frac_logits_sel, expert_frac)

bc_loss = target_ce + frac_ce

n_valid_tokens = int(mask_t.sum().item())
print(f"1. BC Loss (imitation):")
print(f"   target_ce = {target_ce.item():.4f}  (expert chose target {expert_target.item()})")
print(f"   frac_ce   = {frac_ce.item():.4f}  (expert chose {cfg.env.ship_fractions[expert_frac.item()]:.0%})")
print(f"   bc_loss   = {bc_loss.item():.4f}")
print(f"   (For reference, random policy CE ~ -ln(1/{n_valid_tokens-1}) = {-math.log(1/(n_valid_tokens-1)):.2f})")

In [ ]:
# --- 2. PPO Policy Loss (Clipped Surrogate) ---

# Simulate: old policy had log_prob=-2.5, now we have new log_prob
old_log_prob = torch.tensor([-2.5])
new_log_prob = sampled.log_prob

# Advantage: how much better was this action than average?
advantage = torch.tensor([0.8])  # positive = better than expected

# Probability ratio
ratio = (new_log_prob - old_log_prob).exp()

# Clipped surrogate
clip_coef = cfg.ppo.clip_coef  # 0.2
surr1 = ratio * advantage
surr2 = torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef) * advantage
policy_loss = -torch.min(surr1, surr2)  # Negative because we maximize

print(f"2. PPO Policy Loss (Clipped Surrogate):")
print(f"   old_log_prob = {old_log_prob.item():.4f}")
print(f"   new_log_prob = {new_log_prob.item():.4f}")
print(f"   ratio = exp(new - old) = {ratio.item():.4f}")
print(f"   advantage = {advantage.item():.4f}")
print(f"   unclipped = ratio * advantage = {surr1.item():.4f}")
print(f"   clipped   = clamp(ratio, {1-clip_coef}, {1+clip_coef}) * advantage = {surr2.item():.4f}")
print(f"   policy_loss = -min(unclipped, clipped) = {policy_loss.item():.4f}")
clipped = ratio.item() < 1-clip_coef or ratio.item() > 1+clip_coef
print(f"   {'CLIPPED' if clipped else 'NOT CLIPPED'} "
      f"(ratio {'outside' if clipped else 'inside'} [{1-clip_coef:.1f}, {1+clip_coef:.1f}])")

In [ ]:
# --- 3. Value Loss ---
predicted_value = outputs.value  # from CLS token
actual_return = torch.tensor([0.3])  # discounted sum of future rewards

value_loss = 0.5 * (actual_return - predicted_value).pow(2)

print(f"3. Value Loss (MSE):")
print(f"   predicted V(s) = {predicted_value.item():.4f}")
print(f"   actual return  = {actual_return.item():.4f}")
print(f"   value_loss = 0.5 * ({actual_return.item():.4f} - {predicted_value.item():.4f})^2 = {value_loss.item():.6f}")

# --- 4. Entropy Bonus ---
entropy = sampled.entropy
print(f"\n4. Entropy Bonus:")
print(f"   entropy = {entropy.item():.4f}")
print(f"   max possible entropy (uniform over {n_valid_tokens-1} targets) = {math.log(n_valid_tokens-1):.4f}")
print(f"   entropy ratio = {entropy.item() / math.log(n_valid_tokens-1):.2%}")

# --- Total Loss ---
vf_coef = cfg.ppo.vf_coef  # 0.5
ent_coef = cfg.ppo.ent_coef  # 0.01
imitation_coef = 0.3  # mid-decay example

total_loss = (policy_loss.item()
              + vf_coef * value_loss.item()
              - ent_coef * entropy.item()
              + imitation_coef * bc_loss.item())

print(f"\n{'='*60}")
print(f"TOTAL LOSS = policy + {vf_coef}*value - {ent_coef}*entropy + {imitation_coef}*bc")
print(f"           = {policy_loss.item():.4f} + {vf_coef*value_loss.item():.4f} "
      f"- {ent_coef*entropy.item():.4f} + {imitation_coef*bc_loss.item():.4f}")
print(f"           = {total_loss:.4f}")

---

## Part 5: The DAgger Pipeline — Three Phases of Learning

DAgger (Dataset Aggregation) is our secret weapon for bootstrapping the RL agent.
Instead of learning from scratch (which takes millions of steps), we first teach the
agent to imitate an expert, then gradually transition to self-improvement through PPO.

```
Phase 1: Collect Demonstrations     Phase 2: BC Pretrain         Phase 3: PPO + Decaying Imitation

  Apex agent plays 200 games         50 epochs of supervised      PPO training with:
  against random opponent            cross-entropy learning       loss = PPO + beta * BC
  Records (state, action) pairs      on expert demonstrations     beta decays 0.5 -> 0 over 1000 updates
  Maps to our action space           ~493K params updated         Then pure PPO for remaining 4000 updates
```

> **Why apex instead of hybrid?** The apex agent is faster (~1ms/step vs 50-800ms/step
> for hybrid) and serves as a strong benchmark. With 200 games, demo collection takes
> seconds, not hours.

> **Why not just do BC?** Behavioral cloning has a fundamental problem: **compounding
> errors**. The agent never sees states that result from its own mistakes. DAgger
> addresses this by using BC as a starting point, then letting PPO correct through gameplay.

> **Why not just do PPO?** Pure PPO from scratch in a complex strategy game takes
> enormous numbers of games. BC gives it a massive head start.

In [ ]:
# Visualize the imitation coefficient schedule

coef_start = cfg.imitation.coef_start  # 0.5
coef_decay = cfg.imitation.coef_decay_updates  # 1000
total_updates = cfg.ppo.total_updates  # 5000

updates = list(range(0, total_updates + 1, 50))
coefs = [coef_start * max(0.0, 1.0 - u / coef_decay) for u in updates]

print("=== Imitation Coefficient Schedule ===")
print(f"Starts at {coef_start}, decays to 0 over {coef_decay} updates, "
      f"then pure PPO for {total_updates-coef_decay} more\n")

# ASCII chart
width = 60
for i in range(0, len(updates), max(1, len(updates)//20)):
    u = updates[i]
    c = coefs[i]
    bar_len = int(c / coef_start * width)
    bar = '#' * bar_len + '.' * (width - bar_len)
    phase = "BC+PPO" if c > 0 else "Pure PPO"
    print(f"  update {u:>5} | beta={c:.3f} |{bar}| {phase}")

print(f"\nPhase boundaries:")
print(f"  Update 0: BC pretrain ({cfg.imitation.bc_epochs} epochs, lr={cfg.imitation.bc_lr})")
print(f"  Updates 1-{coef_decay}: PPO + imitation (beta: {coef_start} -> 0, lr={cfg.ppo.lr})")
print(f"  Updates {coef_decay+1}-{total_updates}: Pure PPO (lr={cfg.ppo.lr})")

### 5.1 Phase 1: Demonstration Collection

The apex agent plays 200 games against random opponents. For each step, we record
what the expert chose and map it to our action space.

The mapping works by angular matching — if the expert's aim angle is within 90 degrees
of one of our target angles, we record it as that target. Otherwise, NoOp.

In [ ]:
# Demonstrate the action space mapping

print("=== Action Space Mapping ===")
print(f"Expert ({cfg.imitation.bc_expert} agent) outputs: [from_planet_id, angle_radians, num_ships]")
print(f"Our action space: (target_index, fraction_bin)")
print(f"Fraction bins: {cfg.env.ship_fractions}")
print()

# Show how an expert move maps to our space
expert_angle = 1.2  # radians
expert_ships = 30
src_ships = 45

print(f"Example: Expert sends {expert_ships} ships at angle {math.degrees(expert_angle):.1f} deg")
print(f"         from planet with {src_ships} ships\n")

print(f"Step 1: Find closest target by angle (tolerance 90 deg)")
for i, tgt_angle in enumerate(decision.target_angles[:6]):
    diff = abs(expert_angle - tgt_angle)
    diff = min(diff, 2 * math.pi - diff)  # wrap around
    match = ' <-- MATCH' if math.degrees(diff) < 90 else ''
    print(f"  Target {i} angle={math.degrees(tgt_angle):.1f} deg, diff={math.degrees(diff):.1f} deg{match}")

print(f"\nStep 2: Map ship fraction")
frac = expert_ships / src_ships
print(f"  Fraction = {expert_ships}/{src_ships} = {frac:.2f}")
for i, f in enumerate(cfg.env.ship_fractions):
    diff = abs(f - frac)
    best = ' <-- NEAREST' if i == min(range(5), key=lambda j: abs(cfg.env.ship_fractions[j] - frac)) else ''
    print(f"  Bin {i} ({f:.0%}): diff = {diff:.2f}{best}")

### 5.2 The GAE Advantage Computation

PPO needs **advantages** — how much better/worse was this action compared to
the average? We use **Generalized Advantage Estimation (GAE)** which balances
bias and variance through the lambda parameter.

A critical subtlety: **multiple planet decisions share the same reward**. When 3
planets each make a decision on the same turn, they all receive the same step reward.
This means all three decisions are equally credited/blamed.

> **This is a significant weakness**: the credit assignment problem. If planet A
> makes a brilliant move and planet B makes a terrible move on the same turn, they
> both get the same advantage signal.

In [ ]:
# Demonstrate GAE computation

gamma = cfg.ppo.gamma  # 0.99
lam = cfg.ppo.gae_lambda  # 0.95

print("=== GAE (Generalized Advantage Estimation) ===")
print(f"gamma={gamma}, lambda={lam}")
print()

# Simulate a 5-step episode with variable planets per step
rewards = [0.0, 0.0, 0.005, 0.0, 1.0]  # dense + terminal win
values = [0.3, 0.35, 0.4, 0.5, 0.6]     # value estimates
bootstrap_value = 0.0  # terminal state
n_planets_per_step = [2, 3, 2, 1, 2]     # variable number of decisions

print("Step-by-step return computation (backward pass):")
print("-" * 70)

future_return = bootstrap_value
returns = []
advantages = []

for t in range(len(rewards) - 1, -1, -1):
    r = rewards[t]
    done = 1.0 if t == len(rewards) - 1 else 0.0
    future_return = r + gamma * future_return * (1.0 - done)
    adv = future_return - values[t]

    n_planets = n_planets_per_step[t]
    returns.insert(0, future_return)
    advantages.insert(0, adv)

    print(f"  Step {t}: reward={r:.3f}, V(s)={values[t]:.3f}, "
          f"return={future_return:.4f}, advantage={adv:+.4f}, "
          f"assigned to {n_planets} planet decisions")

print(f"\nKey observation: Step 2 had reward=0.005 and 2 planet decisions.")
print(f"Both planets get advantage={advantages[2]:+.4f}, even though one might")
print(f"have been responsible for the good outcome and the other wasn't.")

---

## Part 6: Dense Relative Reward — Learning to Win, Not Just Accumulate

The mixed config uses `dense_relative` reward — a key improvement over the simpler
`sparse` and `dense_absolute` modes. Let's compare all three.

In [ ]:
print("=== Reward Mode Comparison ===\n")

print("1. SPARSE (terminal only):")
print("   reward = +1 (win) or -1 (loss) at the end of the game")
print("   Zero signal for 498 steps. Agent must stumble into good play.")
print()

print("2. DENSE ABSOLUTE:")
print("   reward = delta_own_ships * ship_coef + delta_own_prod * prod_coef")
print("   Rewards accumulating resources, but ignores what enemies are doing.")
print("   Flaw: agent can 'win' the reward game by hoarding, not fighting.")
print()

print("3. DENSE RELATIVE (our config):")
print("   reward = delta(own_ships - best_enemy_ships) * ship_coef")
print("          + delta(own_prod  - best_enemy_prod)  * prod_coef * prod_mult")
print("   Rewards gaining ADVANTAGE over the strongest opponent.")
print("   Capturing an enemy planet is doubly rewarded: +ships for us, -ships for them.")
print()

# Compare reward signals for the same scenario
print("--- Scenario: Capture an enemy prod-3 planet (cost 15 ships, gain planet with 10 ships) ---")
ship_coef = cfg.reward.dense_ship_coef  # 0.002
prod_coef = cfg.reward.dense_prod_coef  # 0.005

# Dense absolute: net -5 ships, +3 prod
dense_abs = (-5) * ship_coef + 3 * prod_coef
# Dense relative: gap changes by +5 ships (we lose 15, enemy loses 10+planet) and +6 prod
dense_rel = 5 * ship_coef + 6 * prod_coef
print(f"  dense_absolute: {dense_abs:+.4f}  (net ship change only)")
print(f"  dense_relative: {dense_rel:+.4f}  (counts both our gain AND enemy's loss)")
print(f"  Relative is {dense_rel/dense_abs:.1f}x stronger signal for attacking!")

### 6.1 Early Production Bonus

In the first ~50 steps, production is worth much more than later. This incentivizes
aggressive early expansion — capturing planets early compounds over the whole game.

```
prod_mult = 1 + early_prod_bonus * max(0, 1 - step / early_prod_bonus_steps)
         = 1 + 9 * max(0, 1 - step / 50)
```
At step 0, production reward is 10x normal. At step 50, it's 1x (normal).

In [ ]:
# Visualize the early production bonus schedule
bonus = cfg.reward.early_prod_bonus  # 9.0
bonus_steps = cfg.reward.early_prod_bonus_steps  # 50

print("=== Early Production Bonus Schedule ===")
print(f"Formula: prod_mult = 1 + {bonus} * max(0, 1 - step/{bonus_steps})\n")

step_points = list(range(0, 101, 5))
for step in step_points:
    t = max(0.0, 1.0 - step / bonus_steps)
    mult = 1.0 + bonus * t
    bar = '#' * int(mult * 4)
    print(f"  step {step:>3}: prod_mult = {mult:>5.1f}x  {bar}")

print(f"\nCapturing a prod-3 planet at different times:")
for step in [0, 10, 25, 50, 100]:
    t = max(0.0, 1.0 - step / bonus_steps)
    mult = 1.0 + bonus * t
    reward_prod = 3 * prod_coef * mult
    print(f"  step {step:>3}: prod reward = 3 * {prod_coef} * {mult:.1f} = {reward_prod:.4f}")

### 6.2 Real Reward Signal

Let's compute actual dense_relative rewards from real environment steps.

In [ ]:
# Run a few steps of the real environment to compute actual rewards
# Reset for fresh reward tracking
env2 = make("orbit_wars", debug=False)
env2.reset(num_agents=2)
states2 = env2.step([[], []])

# Track ships and production gap — use parse_observation for proper field access
def count_player_parsed(raw_obs, player):
    gs = parse_observation(raw_obs)
    ships = sum(p.ships for p in gs.planets if p.owner == player)
    ships += sum(f.ships for f in gs.fleets if f.owner == player)
    prod = sum(p.production for p in gs.planets if p.owner == player)
    return ships, prod

prev_ship_gap, prev_prod_gap = 0.0, 0.0
p0_obs = states2[0].observation
own_s, own_p = count_player_parsed(p0_obs, 0)
other_s, other_p = count_player_parsed(p0_obs, 1)
prev_ship_gap = own_s - other_s
prev_prod_gap = own_p - other_p

print("=== Real Dense Relative Rewards (first 30 steps) ===")
print(f"{'Step':>4} {'Ship Gap':>10} {'Prod Gap':>10} {'Reward':>10}")
print("-" * 40)

for step_i in range(30):
    p0_moves = apex_agent(states2[0].observation)
    p1_moves = apex_agent(states2[1].observation)
    states2 = env2.step([p0_moves, p1_moves])

    p0_obs = states2[0].observation
    own_s, own_p = count_player_parsed(p0_obs, 0)
    other_s, other_p = count_player_parsed(p0_obs, 1)

    ship_gap = own_s - other_s
    prod_gap = own_p - other_p

    delta_ship_gap = ship_gap - prev_ship_gap
    delta_prod_gap = prod_gap - prev_prod_gap

    # Early production bonus
    t = max(0.0, 1.0 - p0_obs.step / bonus_steps)
    pmult = 1.0 + bonus * t

    reward = delta_ship_gap * ship_coef + delta_prod_gap * prod_coef * pmult

    if abs(reward) > 0.0001:
        print(f"{p0_obs.step:>4} {ship_gap:>+10.0f} {prod_gap:>+10.0f} {reward:>+10.4f}")

    prev_ship_gap = ship_gap
    prev_prod_gap = prod_gap

---

## Part 7: Mixed Self-Play and 4-Player Training Curriculum

The `transformer_mixed` config introduces two powerful training innovations:

1. **MixedScheduler**: Blends rule-based opponents (apex) with self-play, gradually
   shifting from mostly rule-based to mostly self-play
2. **4-player games**: 30% of training episodes use 4 players instead of 2

In [ ]:
print("=== MixedScheduler Configuration ===\n")
print(f"  four_player_prob: {cfg.four_player_prob} (30% of episodes are 4-player)")
print(f"  rule_based_prob_start: {cfg.rule_based_prob_start} (100% apex at start)")
print(f"  rule_based_prob_end: {cfg.rule_based_prob_end} (20% apex at end)")
print(f"  rule_based_decay_updates: {cfg.rule_based_decay_updates}")
print(f"  self_play_update_interval: {cfg.self_play_update_interval} (sync weights every N updates)")
print()

# Visualize the opponent mix over training
print("=== Opponent Mix Schedule ===\n")
decay = cfg.rule_based_decay_updates
rb_start = cfg.rule_based_prob_start
rb_end = cfg.rule_based_prob_end

width = 50
for update in range(0, cfg.ppo.total_updates + 1, 250):
    frac = min(1.0, update / decay) if decay > 0 else 1.0
    rb_prob = rb_start + frac * (rb_end - rb_start)
    sp_prob = 1.0 - rb_prob

    rb_bar = '#' * int(rb_prob * width)
    sp_bar = '~' * int(sp_prob * width)
    print(f"  update {update:>5} | apex={rb_prob:.0%} self={sp_prob:.0%} |{rb_bar}{sp_bar}|")

print(f"\nLegend: # = rule-based (apex), ~ = self-play")

### 7.1 How MixedScheduler Works

For each episode, the scheduler:
1. Decides if it's a 2p or 4p game (30% chance of 4p)
2. For each opponent slot (1 in 2p, 3 in 4p), independently decides rule-based vs self-play
3. Self-play opponents get periodic weight syncs from the training policy

```python
class MixedScheduler:
    def sample_episode(self):
        is_4p = random.random() < self.cfg.four_player_prob
        n_opp = 3 if is_4p else 1
        rb_prob = self._rule_based_prob()
        opponents = []
        for _ in range(n_opp):
            if random.random() < rb_prob:
                opponents.append(self.rule_based)   # apex
            else:
                opponents.append(self.self_play)     # copy of current policy
        return (4 if is_4p else 2), opponents
```

In [ ]:
# Demonstrate MixedScheduler behavior at different training stages
import random as rng
rng.seed(42)

print("=== MixedScheduler Episode Sampling ===\n")

for update in [0, 500, 1000, 2000, 5000]:
    frac = min(1.0, update / decay) if decay > 0 else 1.0
    rb_prob = rb_start + frac * (rb_end - rb_start)

    episodes = []
    for _ in range(20):
        is_4p = rng.random() < cfg.four_player_prob
        n_opp = 3 if is_4p else 1
        opps = sum(1 for _ in range(n_opp) if rng.random() < rb_prob)
        mode = f"{'4p' if is_4p else '2p'}"
        episodes.append(f"{mode}({'A'*opps}{'S'*(n_opp-opps)})")

    print(f"  Update {update:>5} (rb={rb_prob:.0%}): {' '.join(episodes[:10])}...")

print(f"\nKey: 2p/4p = players, A = apex, S = self-play")

### 7.2 Self-Play Weight Syncing

The self-play opponent maintains a **separate copy** of the transformer policy.
Every `self_play_update_interval` (50) updates, the training policy's weights are
copied to the self-play opponent.

```python
class SelfPlayOpponent:
    def sync_from(self, source_policy: TransformerPolicy):
        self.policy.load_state_dict(source_policy.state_dict())
        self.policy.eval()
```

This means the self-play opponent is always slightly "behind" the training policy,
providing a curriculum of increasingly strong opponents.

### 7.3 Why This Curriculum Works

```
Early training (updates 0-1000):
  - Mostly apex opponents (100% -> 60%)
  - Learn basic strategy: capture planets, defend, attack
  - BC imitation loss guides toward expert behavior
  - Agent learns "how to play the game"

Mid training (updates 1000-2000):
  - Decreasing apex, increasing self-play (60% -> 20%)
  - Discover strategies that beat yourself (exploration)
  - No more imitation loss — pure PPO
  - Agent learns "how to beat diverse opponents"

Late training (updates 2000-5000):
  - Mostly self-play (80%), some apex (20%)
  - Arms race: each improvement makes the opponent stronger too
  - Prevents overfitting to one opponent style
  - Agent learns "how to be robust"
```

**4-player dynamics**: In 4-player games, the optimal strategy changes dramatically.
You can't just focus on one opponent — you need to manage multiple threats, form
temporary alliances of inaction, and strike weakened opponents after they fight each other.

---

## Part 8: Putting It All Together — A Complete Training Step

Let's simulate one complete training iteration to see how all the pieces fit together.

In [ ]:
# Simulate a mini rollout to show the full training loop
policy.train()
optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.ppo.lr)

print("=== Simulated Training Step ===")
print()

# 1. Collect rollout data
n_decisions = len(my_planets)  # one decision per owned planet
all_features = []
all_actions = []
all_log_probs = []
all_values = []

# Reset transit for clean collection
transit = compute_fleet_transit(state)

print("Step 1: Collect rollout")
for i, src_planet in enumerate(my_planets[:n_decisions]):
    dec = encode_source_decision(src_planet, state, transit, cfg.env)

    feats = {
        'global': torch.tensor(dec.global_features, dtype=torch.float32).unsqueeze(0),
        'src_s': torch.tensor(dec.source_scalars, dtype=torch.float32).unsqueeze(0),
        'src_p': torch.tensor(dec.source_position, dtype=torch.float32).unsqueeze(0),
        'knn_s': torch.tensor(dec.knn_scalars, dtype=torch.float32).unsqueeze(0),
        'knn_p': torch.tensor(dec.knn_positions, dtype=torch.float32).unsqueeze(0),
        'tgt_s': torch.tensor(dec.target_scalars, dtype=torch.float32).unsqueeze(0),
        'tgt_p': torch.tensor(dec.target_positions, dtype=torch.float32).unsqueeze(0),
        'mask': torch.tensor(dec.target_mask, dtype=torch.bool).unsqueeze(0),
    }

    with torch.no_grad():
        out = policy(feats['global'], feats['src_s'], feats['src_p'],
                     feats['knn_s'], feats['knn_p'],
                     feats['tgt_s'], feats['tgt_p'], feats['mask'])
        act = sample_actions(out, deterministic=False)

    all_features.append(feats)
    all_actions.append(act)
    all_log_probs.append(act.log_prob.item())
    all_values.append(out.value.item())

    tgt_name = 'NoOp' if act.target_index.item() == 0 else f'Target {act.target_index.item()}'
    frac = cfg.env.ship_fractions[act.fraction_bin.item()]
    print(f"  Decision {i}: Planet {src_planet.id} -> {tgt_name}, "
          f"fraction={frac:.0%}, log_prob={act.log_prob.item():.3f}, V={out.value.item():.3f}")

# Simulate rewards
rewards_sim = [0.0] * n_decisions
returns_sim = [0.3 + i * 0.02 for i in range(n_decisions)]
advantages_sim = [r - v for r, v in zip(returns_sim, all_values)]

print(f"\n  Rewards: {rewards_sim}")
print(f"  Returns (GAE): {[f'{r:.3f}' for r in returns_sim]}")
print(f"  Advantages: {[f'{a:+.3f}' for a in advantages_sim]}")

In [ ]:
# 2. PPO update on collected data
print("Step 2: PPO update")
print(f"  Epochs: {cfg.ppo.epochs}")
print(f"  Clip coefficient: {cfg.ppo.clip_coef}")

# Normalize advantages
adv_tensor = torch.tensor(advantages_sim, dtype=torch.float32)
adv_norm = (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)
print(f"  Raw advantages: {[f'{a:+.3f}' for a in advantages_sim]}")
print(f"  Normalized:     {[f'{a.item():+.3f}' for a in adv_norm]}")

returns_tensor = torch.tensor(returns_sim, dtype=torch.float32)

# One epoch of PPO
total_policy_loss = 0
total_value_loss = 0
total_entropy = 0

for i in range(n_decisions):
    feats = all_features[i]
    old_act = all_actions[i]
    old_lp = all_log_probs[i]

    out = policy(feats['global'], feats['src_s'], feats['src_p'],
                 feats['knn_s'], feats['knn_p'],
                 feats['tgt_s'], feats['tgt_p'], feats['mask'])

    new_lp, ent = action_log_prob_and_entropy(
        out, old_act.target_index, old_act.fraction_bin
    )

    ratio = (new_lp - old_lp).exp()
    a = adv_norm[i]
    surr1 = ratio * a
    surr2 = torch.clamp(ratio, 1 - cfg.ppo.clip_coef, 1 + cfg.ppo.clip_coef) * a
    pol_loss = -torch.min(surr1, surr2)

    val_loss = 0.5 * (returns_tensor[i] - out.value).pow(2)

    total_policy_loss += pol_loss.item()
    total_value_loss += val_loss.item()
    total_entropy += ent.item()

print(f"\n  Avg policy loss: {total_policy_loss/n_decisions:.4f}")
print(f"  Avg value loss:  {total_value_loss/n_decisions:.4f}")
print(f"  Avg entropy:     {total_entropy/n_decisions:.4f}")
print(f"\n  (In real training, this runs for {cfg.ppo.epochs} epochs")
print(f"   over {cfg.ppo.rollout_steps * cfg.ppo.num_envs} decisions per update)")

# 3. Self-play sync
print(f"\nStep 3: Self-play opponent sync")
print(f"  Every {cfg.self_play_update_interval} updates, copy training weights to self-play opponent")
print(f"  MixedScheduler picks opponents per episode (see Part 7)")

---

## Part 9: Strengths, Weaknesses, and the Path Forward

Now that we've seen every component, let's assess the architecture honestly.

In [ ]:
print("""
=== ARCHITECTURE STRENGTHS ===

1. STRONG INDUCTIVE BIASES
   - Per-planet sequential decisions mirror human reasoning
   - Transit state tracking prevents redundant fleet launches
   - Sun avoidance baked into the mask (impossible to send through sun)
   - Targets sorted by distance (nearby = more relevant)

2. EFFICIENT ARCHITECTURE
   - ~493K params is tiny by modern standards
   - 2-layer transformer is fast (~1ms inference)
   - Factored actions reduce action space from T*5 to T + 5
   - Shared position encoder reduces parameters

3. DENSE RELATIVE REWARD
   - Rewards gaining advantage, not just accumulating ships
   - Early production bonus incentivizes fast expansion
   - Much stronger learning signal than sparse reward

4. MIXED TRAINING CURRICULUM
   - DAgger bootstrap: expert knowledge + RL improvement
   - Self-play prevents overfitting to one opponent
   - 4-player training teaches multi-threat awareness
   - Rule-based decay provides a smooth curriculum

=== ARCHITECTURE WEAKNESSES ===

1. CREDIT ASSIGNMENT
   - All planets in a turn share the same reward signal
   - No per-planet advantage estimation
   - Fleet results arrive many steps after launch decisions

2. SEQUENTIAL DECISIONS ARE ORDER-DEPENDENT
   - Processing planets by ship count (desc) is an arbitrary choice
   - The first planet gets the "cleanest" transit state
   - No mechanism to revise earlier decisions

3. SINGLE-STEP HORIZON
   - Each forward pass decides for ONE turn only
   - No explicit multi-turn planning
   - The value function must implicitly encode future plans

4. LIMITED ACTION SPACE
   - Only 5 fraction bins (20/40/60/80/100%)
   - Only the 30 nearest planets as targets
   - Only one fleet per source planet per turn
""")

### 9.1 Paths to Improvement

In [ ]:
improvements = [
    ("HIGH IMPACT / LOW EFFORT", [
        ("Larger model",
         "embed_dim=256, n_layers=4, ff_dim=512 (~2M params). With GPU training"
         " this is basically free. More capacity for strategic reasoning.",
         "configs: model.embed_dim: 256, model.n_layers: 4"),
        ("Reward shaping: fleet launch credit",
         "Give immediate positive reward when launching a fleet toward"
         " an enemy/neutral planet. Addresses the 'negative reward for attacking' problem.",
         "Modify src/env.py: _compute_reward()"),
        ("Comet tracking features",
         "Comets spawn at predictable intervals with known paths. Add comet"
         " position predictions to target features. Free production!",
         "Modify src/features.py"),
    ]),
    ("HIGH IMPACT / MEDIUM EFFORT", [
        ("Per-planet value heads (multi-agent credit assignment)",
         "Instead of one value from CLS, have each target token predict its own"
         " advantage. Each planet's decision gets its own baseline.",
         "Modify src/policy.py + src/ppo.py"),
        ("Population-based training (league)",
         "Maintain a pool of 5-10 past checkpoints. Sample opponents from the"
         " pool. Prevents forgetting old strategies.",
         "Modify src/train.py + src/opponents.py"),
    ]),
    ("HIGH IMPACT / HIGH EFFORT", [
        ("Attention-based multi-planet coordination",
         "Instead of sequential per-planet decisions, use a second transformer"
         " that attends over all owned planets simultaneously.",
         "Major redesign of src/policy.py + src/train.py"),
        ("Meta-learning (RL^2 with GRU)",
         "Add a recurrent component that persists across turns within a game."
         " The hidden state becomes an implicit opponent model.",
         "Modify src/policy.py: add GRU state"),
    ]),
]

for category, items in improvements:
    print(f"\n{'='*70}")
    print(f"  {category}")
    print(f"{'='*70}")
    for name, desc, where in items:
        print(f"\n  >> {name}")
        print(f"     {desc}")
        print(f"     Where: {where}")

---

## Part 10: Key Takeaways

### What We Built

A **Transformer Mixed PPO agent** that:
1. Parses raw game observations into structured features with domain knowledge
2. Encodes those features into a token sequence (CLS + NoOp + Targets)
3. Processes tokens through a 2-layer transformer with masked attention
4. Outputs factored actions (target selection + ship fraction) and a state value
5. Learns via 3-phase DAgger: BC pretrain -> PPO + imitation decay -> Pure PPO
6. Uses dense relative reward with early production bonus for stronger learning signal
7. Trains against a mix of rule-based and self-play opponents in 2p and 4p games

### The Big Ideas

1. **Feature engineering dominates**: The 80+ features per decision encode decades of
   game AI wisdom (transit prediction, sun avoidance, orbit detection).

2. **Inductive biases are free performance**: Per-planet decisions, sorted targets,
   shared position encoders — each constrains the model but reduces complexity.

3. **DAgger bridges the expert-RL gap**: BC gives a head start, PPO goes beyond.

4. **Dense relative reward > sparse**: Rewarding advantage over opponents gives a
   learning signal at every step, not just at game end.

5. **Mixed self-play prevents overfitting**: Gradually shifting from expert opponents
   to self-play creates a natural curriculum.

6. **Credit assignment is the bottleneck**: The biggest weakness isn't the architecture
   — it's that multiple decisions share rewards.

In [ ]:
# Final summary: model card
print("+" + "="*62 + "+")
print("|              TRANSFORMER MIXED PPO -- MODEL CARD             |")
print("+" + "="*62 + "+")
print(f"|  Parameters:         {count_params(policy):>10,}                        |")
print(f"|  Embedding dim:      {cfg.model.embed_dim:>10}                        |")
print(f"|  Transformer layers: {cfg.model.n_layers:>10}                        |")
print(f"|  Attention heads:    {cfg.model.n_heads:>10}                        |")
print(f"|  FF dim:             {cfg.model.ff_dim:>10}                        |")
print(f"|  Max targets:        {cfg.env.max_targets:>10}                        |")
print(f"|  Fraction bins:      {len(cfg.env.ship_fractions):>10}                        |")
print(f"|  Reward mode:        {'dense_relative':>14}                    |")
print(f"|  Early prod bonus:   {cfg.reward.early_prod_bonus:>10}x (step 0)              |")
print(f"|  BC expert:          {cfg.imitation.bc_expert:>10}                        |")
print(f"|  BC games:           {cfg.imitation.bc_games:>10}                        |")
print(f"|  BC epochs:          {cfg.imitation.bc_epochs:>10}                        |")
print(f"|  PPO updates:        {cfg.ppo.total_updates:>10}                        |")
print(f"|  4-player prob:      {cfg.four_player_prob:>10}                        |")
print(f"|  Rule-based decay:   {cfg.rule_based_decay_updates:>10} updates                 |")
print(f"|  Imitation decay:    {cfg.imitation.coef_decay_updates:>10} updates                 |")
print("+" + "="*62 + "+")